# Hyperparameter Sweep — Quantum Grant Matcher

**InclusiFund Research Vault**
**Status:** Experimental
**Data:** SYNTHETIC ONLY — no client data

---

## Objective

Sweep across layers, learning rate, and batch size to find the optimal
configuration for the variational quantum circuit grant matcher.

**Grid:**
- Layers: [1, 2, 3]
- Learning rate: [0.01, 0.05, 0.1]
- Batch size: [8, 16, 32]

Total combinations: 27

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools
import time

from quantum_grants.training.trainer import train, TrainingConfig

print('Modules loaded successfully')

## 1. Define Hyperparameter Grid

In [ ]:
layers_grid = [1, 2, 3]
lr_grid = [0.01, 0.05, 0.1]
batch_grid = [8, 16, 32]

combos = list(itertools.product(layers_grid, lr_grid, batch_grid))
print(f'Total configurations: {len(combos)}')
for i, (l, lr, bs) in enumerate(combos[:5]):
    print(f'  {i+1}. layers={l}, lr={lr}, batch={bs}')
print('  ...')

## 2. Run Sweep

In [ ]:
results = []
all_logs = {}

for i, (n_layers, lr, batch_size) in enumerate(combos):
    config = TrainingConfig(
        n_layers=n_layers,
        learning_rate=lr,
        n_epochs=15,
        batch_size=batch_size,
        n_grants=10,
        n_applicants=20,
        seed=42,
    )
    
    t0 = time.time()
    logs = train(config)
    elapsed = time.time() - t0
    
    final = logs[-1]
    key = f'L{n_layers}_lr{lr}_bs{batch_size}'
    all_logs[key] = logs
    
    results.append({
        'layers': n_layers,
        'learning_rate': lr,
        'batch_size': batch_size,
        'final_loss': final.loss,
        'final_accuracy': final.accuracy,
        'time_s': elapsed,
        'key': key,
    })
    
    print(f'[{i+1}/{len(combos)}] {key}: loss={final.loss:.4f}, acc={final.accuracy:.4f}, time={elapsed:.1f}s')

print(f'\nSweep complete: {len(results)} configurations tested')

## 3. Results Table

In [ ]:
df = pd.DataFrame(results).sort_values('final_accuracy', ascending=False)
df = df.reset_index(drop=True)
df.index += 1
df.index.name = 'rank'

print('Top 10 configurations by accuracy:')
print(df[['layers', 'learning_rate', 'batch_size', 'final_loss', 'final_accuracy', 'time_s']].head(10).to_string())
print(f'\nBest: {df.iloc[0]["key"]} — accuracy={df.iloc[0]["final_accuracy"]:.4f}')

## 4. Heatmap — Accuracy by Layers vs Learning Rate

In [ ]:
# Average accuracy across batch sizes for each (layers, lr) pair
pivot = pd.DataFrame(results).groupby(['layers', 'learning_rate'])['final_accuracy'].mean().unstack()

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{x}' for x in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f'{y} layers' for y in pivot.index])
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Layers')
ax.set_title('Mean Accuracy by Layers vs Learning Rate\n(averaged over batch sizes)')

# Annotate cells
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i, j]:.3f}', ha='center', va='center', fontsize=12, fontweight='bold')

plt.colorbar(im, label='Accuracy')
plt.tight_layout()
plt.show()

## 5. Training Curves — Top 3 Configurations

In [ ]:
top3 = df.head(3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for _, row in top3.iterrows():
    key = row['key']
    logs = all_logs[key]
    epochs = [l.epoch for l in logs]
    ax1.plot(epochs, [l.loss for l in logs], '-o', markersize=3, label=key)
    ax2.plot(epochs, [l.accuracy for l in logs], '-o', markersize=3, label=key)

ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss — Top 3')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training Accuracy — Top 3')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Quantum Grant Matcher — Top 3 Hyperparameter Configs', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

Key findings from the hyperparameter sweep:

1. **Layers** — more layers generally improve expressivity but increase training time
2. **Learning rate** — moderate rates (0.05) tend to balance convergence speed and stability
3. **Batch size** — smaller batches add noise that can help escape local minima

The optimal configuration will be used as the default for production quantum matching.

---
*Next: Compare best quantum config against classical baselines in notebook 03.*